In [ ]:
!pip uninstall -y delta-spark pyspark
!pip install pyspark==4.0.1 delta-spark==4.0.1

Found existing installation: pyspark 4.0.4
Uninstalling pyspark-4.0.4:
  Successfully uninstalled pyspark-4.0.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.2/434.2 MB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 2.5 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-4.0.1-py2.py3-none-any.whl size=434813860 sha256=4dbd7879b295fd45c4e0ef0c5d8a0e160ec30d7c0a5436ef3f9b2158482f8366
  Stored in directory: /root/.cache/pip/wheels/00/e3/92/8594f4cee2c9fd4ad82fe85e4bf2559ab8ea84ef19b1dd3d15
Successfully built pyspark


In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as sf
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("UPI Payment Data Engineering")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

#*Step 1: Creating the initial dataset*

In [ ]:
claims_data = [
    (" CLM1001 ", " cust1001 ", " hospitalisation ", "₹1,25,000", " 2026-01-05 ", "approved", "Dr. Sharma", "PAT10001"),
    ("CLM1002", "CUST1002", "ACCIDENT", "$2,500", "2026/01/07", " APPROVED ", "Dr. Mehta", "PAT10002"),
    ("clm1003", "cust1003", "Vehicle Damage", "75000", "08-01-2026", "Pending", "Dr. Roy", "PAT10003"),
    ("CLM1004 ", " CUST1004", "THEFT", "₹85K", "2026-01-10", "Rejected", "Dr. Khan", "PAT10004"),
    ("CLM1005", "CUST1005 ", "Maternity", "1.5L", "2026-01-11", "approved", "Dr. Das", "PAT10005"),
    ("CLM1006", "cust1006", "HOSPITALIZATION", "₹2,00,000", "2026-01-12", "Approved", "Dr. Sharma", "PAT10006"),
    ("CLM1007", "CUST1007", "accident ", "35000", "2026-01-13", "pending", "Dr. Mehta", "PAT10007"),
    ("CLM1008", "CUST1008", "THEFT", "INVALID", "2026-01-14", "Approved", "Dr. Roy", "PAT10008"),
    ("CLM1009", "CUST1009", "DEATH", "₹5L", "2026-01-15", "APPROVED", "Dr. Khan", "PAT10009"),
    ("CLM1010", "CUST1010", "OPD", "₹15,000", "2026-01-16", "approved", "Dr. Das", "PAT10010"),
    ("CLM1011", "CUST1011", "MATURITY", "2L", "2026-01-17", "Pending", "Dr. Sharma", "PAT10011"),
    ("CLM1012", "CUST1012", "SURGERY", "₹95,000", "2026-01-18", "approved", "Dr. Mehta", "PAT10012"),
    ("CLM1012", "CUST1012", "SURGERY", "₹95,000", "2026-01-18", "approved", "Dr. Mehta", "PAT10012"),
    ("CLM1013", "CUST1013", "FLIGHT DELAY", "€1200", "2026-01-19", "Approved", "Dr. Roy", "PAT10013"),
    ("CLM1014", "CUST1014", "BAGGAGE LOSS", "45000", "2026-01-20", "Rejected", "Dr. Khan", "PAT10014"),

    # --- NEW MESSY RECORDS ---

    (" CLM1015", "cust1015", "hospitalisation", " ₹75,500 ", "2026-01-21", " APPROVED", "dr. sharma", " PAT10015 "),
    ("CLM1016 ", " CUST1016 ", "Accident", "Rs. 1,20,000", "21/01/2026", "approved ", "Dr Mehta", "PAT10016"),
    ("clm1017", "cust1017", "vehicle damage", "$3,500.75", "2026-01-22", "PENDING", "Dr. Roy ", "PAT10017"),
    ("CLM1018", "CUST1018", " Theft ", "85000 INR", "2026-01-23", " rejected ", "Dr. Khan", "PAT10018"),
    ("CLM1019", "CUST1019", "maternity", "₹1.25L", "2026-01-24", "Approved", "Dr. Das", "PAT10019"),
    ("CLM1020", "CUST1020", "HOSPITALIZATION", "200000", "2026-01-25", "approved", "Dr. Sharma", "PAT10020"),

    ("CLM1021", "CUST1021", "accident", "₹ 45,000", "2026-01-26", "Pending", "Dr. Mehta", "PAT10021"),
    ("CLM1022", "CUST1022", "DEATH", "5.5L", "2026-01-27", "APPROVED", "Dr. Roy", "PAT10022"),
    ("CLM1023", "CUST1023", "OPD", "15000 INR", "2026-01-28", "Approved", "Dr. Khan", "PAT10023"),
    ("CLM1024", "CUST1024", "surgery ", "₹90,000/-", "2026-01-29", "approved", "Dr. Das", "PAT10024"),

    ("CLM1025", "CUST1025", "FLIGHT_DELAY", "€1,500", "2026-01-30", "Pending", "Dr. Roy", "PAT10025"),
    ("CLM1026", "CUST1026", "BAGGAGE LOSS", "₹35K", "2026-01-31", "Rejected", "Dr. Khan", "PAT10026"),
    ("CLM1027", "CUST1027", "MOTOR INSURANCE", "₹2.5L", "2026-02-01", "Approved", "Dr. Sharma", "PAT10027"),
    ("CLM1028", "CUST1028", "vehicle damage ", "1.2L", "2026-02-02", "approved", "Dr. Mehta", "PAT10028"),

    ("CLM1029", "CUST1029", "HEALTH INSURANCE", "₹1,10,000", "2026-02-03", "Pending", "Dr. Roy", "PAT10029"),
    ("CLM1030", "CUST1030", "Maternity", "₹2L", "2026-02-04", "Approved", "Dr. Das", "PAT10030"),

    # --- ID PROBLEMS ---

    (" CLM1031 ", " CUST1031 ", "ACCIDENT", "55000", "2026-02-05", "approved", "Dr. Sharma", "PAT10031"),
    ("CLM1032", "cust-1032", "THEFT", "₹60K", "2026-02-06", "Approved", "Dr. Khan", "PAT10032"),
    ("CLM1033A", "CUST1033", "OPD", "12000", "2026-02-07", "Pending", "Dr. Das", "PAT10033"),
    ("CLM1034", "CUST_1034", "SURGERY", "₹80,000", "2026-02-08", "approved", "Dr. Mehta", "PAT10034"),
    ("CLM1035", "CUST1035", "DEATH", "₹4L", "2026-02-09", "Rejected", "Dr. Roy", "PAT-10035"),

    # --- DATE PROBLEMS ---

    ("CLM1036", "CUST1036", "ACCIDENT", "₹35,000", "2026/02/10", "Approved", "Dr. Sharma", "PAT10036"),
    ("CLM1037", "CUST1037", "THEFT", "45000", "10-02-2026", "approved", "Dr. Khan", "PAT10037"),
    ("CLM1038", "CUST1038", "MATURITY", "1.8L", "11/02/26", "Pending", "Dr. Das", "PAT10038"),
    ("CLM1039", "CUST1039", "OPD", "₹12,500", "2026.02.12", "APPROVED", "Dr. Roy", "PAT10039"),
    ("CLM1040", "CUST1040", "SURGERY", "₹95,000", "not available", "Approved", "Dr. Mehta", "PAT10040"),

    # --- CLAIM TYPE PROBLEMS ---

    ("CLM1041", "CUST1041", "hospitalization", "₹1L", "2026-02-14", "approved", "Dr. Sharma", "PAT10041"),
    ("CLM1042", "CUST1042", "Hospitalisation", "₹1.2L", "2026-02-15", "Approved", "Dr. Sharma", "PAT10042"),
    ("CLM1043", "CUST1043", "vehicle damage", "₹75K", "2026-02-16", "Pending", "Dr. Roy", "PAT10043"),
    ("CLM1044", "CUST1044", "VEHICLE-DAMAGE", "80000", "2026-02-17", "approved", "Dr. Khan", "PAT10044"),
    ("CLM1045", "CUST1045", "FLIGHT DELAY ", "€900", "2026-02-18", "Rejected", "Dr. Das", "PAT10045"),

    # --- STATUS PROBLEMS ---

    ("CLM1046", "CUST1046", "THEFT", "₹55K", "2026-02-19", "APPROVE", "Dr. Khan", "PAT10046"),
    ("CLM1047", "CUST1047", "ACCIDENT", "₹70K", "2026-02-20", "REJ", "Dr. Mehta", "PAT10047"),
    ("CLM1048", "CUST1048", "OPD", "15000", "2026-02-21", "In Review", "Dr. Roy", "PAT10048"),
    ("CLM1049", "CUST1049", "SURGERY", "₹90K", "2026-02-22", "pending ", "Dr. Das", "PAT10049"),
    ("CLM1050", "CUST1050", "DEATH", "₹6L", "2026-02-23", "unknown", "Dr. Sharma", "PAT10050"),

    # --- NULL / EMPTY / BAD VALUES ---

    (None, "CUST1051", "ACCIDENT", "₹45K", "2026-02-24", "Approved", "Dr. Mehta", "PAT10051"),
    ("CLM1052", None, "THEFT", "₹50K", "2026-02-25", "Pending", "Dr. Khan", "PAT10052"),
    ("CLM1053", "CUST1053", None, "₹75K", "2026-02-26", "Approved", "Dr. Roy", "PAT10053"),
    ("CLM1054", "CUST1054", "OPD", None, "2026-02-27", "approved", "Dr. Das", "PAT10054"),
    ("CLM1055", "CUST1055", "SURGERY", "₹85K", None, "Pending", "Dr. Sharma", "PAT10055"),
    ("", "CUST1056", "MATURITY", "1L", "2026-03-01", "Approved", "Dr. Mehta", "PAT10056"),

    # --- AMOUNT PROBLEMS ---

    ("CLM1057", "CUST1057", "ACCIDENT", "₹1,25,500.50", "2026-03-02", "Approved", "Dr. Roy", "PAT10057"),
    ("CLM1058", "CUST1058", "THEFT", "INR 75000", "2026-03-03", "approved", "Dr. Khan", "PAT10058"),
    ("CLM1059", "CUST1059", "OPD", "Rs 12,500", "2026-03-04", "Pending", "Dr. Das", "PAT10059"),
    ("CLM1060", "CUST1060", "SURGERY", "75K INR", "2026-03-05", "Approved", "Dr. Sharma", "PAT10060"),
    ("CLM1061", "CUST1061", "DEATH", "₹7.5 Lakh", "2026-03-06", "Approved", "Dr. Mehta", "PAT10061"),
    ("CLM1062", "CUST1062", "ACCIDENT", "1,25,000", "2026-03-07", "Pending", "Dr. Roy", "PAT10062"),
    ("CLM1063", "CUST1063", "THEFT", "N/A", "2026-03-08", "Rejected", "Dr. Khan", "PAT10063"),

    # --- DOCTOR NAME PROBLEMS ---

    ("CLM1064", "CUST1064", "OPD", "₹10K", "2026-03-09", "Approved", " Dr. Sharma ", "PAT10064"),
    ("CLM1065", "CUST1065", "ACCIDENT", "₹40K", "2026-03-10", "approved", "DR SHARMA", "PAT10065"),
    ("CLM1066", "CUST1066", "THEFT", "₹60K", "2026-03-11", "Pending", "dr. mehta", "PAT10066"),
    ("CLM1067", "CUST1067", "SURGERY", "₹95K", "2026-03-12", "Approved", "DR.ROY", "PAT10067"),
    ("CLM1068", "CUST1068", "MATERNITY", "₹1.5L", "2026-03-13", "approved", None, "PAT10068"),

    # --- MORE DUPLICATE / NEAR-DUPLICATE CASES ---

    ("CLM1069", "CUST1069", "ACCIDENT", "₹50K", "2026-03-14", "Approved", "Dr. Sharma", "PAT10069"),
    ("CLM1069 ", " CUST1069", " accident ", "₹50K", "2026-03-14", " approved ", "Dr. Sharma", "PAT10069"),

    ("CLM1070", "CUST1070", "THEFT", "₹65K", "2026-03-15", "Pending", "Dr. Khan", "PAT10070"),
    ("clm1070", "cust1070", "THEFT", "65000", "15-03-2026", "pending", "Dr Khan", "PAT10070"),

    # --- EXTREME STRING MESS ---

    ("  clm1071  ", "  cust1071  ", "  accident  ", " ₹ 55,000/- ", " 2026-03-16 ", " APPROVED ", " Dr. Mehta ", " PAT10071 "),
    ("CLM1072", "CUST1072", "theft!!!", "₹60,000 only", "2026-03-17", "approved", "Dr. Roy", "PAT10072"),
    ("CLM1073", "CUST1073", "Accident ", "$ 2,500 USD", "2026-03-18", "PENDING", "DR. KHAN", "PAT10073"),
    ("CLM1074", "CUST1074", "hospitalization ", "Rs.1,50,000/-", "2026-03-19", "Approved", "Dr.Das", "PAT10074"),
    (" CLM1075", "CUST1075 ", "  MATURITY  ", "2.5 L", "2026-03-20", "approved ", "Dr. Sharma", "PAT10075"),
]

In [ ]:
columns = ["Claim_ID", "Customer_ID", "Claim_Type", "Claim_Amount",
           "Claim_Date", "Claim_Status", "Doctor_Name", "Patient_ID"]

In [ ]:
dirty_claims_df = spark.createDataFrame(data=claims_data, schema=columns)

In [ ]:
dirty_claims_df.show()

+---------+-----------+-----------------+------------+------------+------------+-----------+----------+
| Claim_ID|Customer_ID|       Claim_Type|Claim_Amount|  Claim_Date|Claim_Status|Doctor_Name|Patient_ID|
+---------+-----------+-----------------+------------+------------+------------+-----------+----------+
| CLM1001 |  cust1001 | hospitalisation |   ₹1,25,000| 2026-01-05 |    approved| Dr. Sharma|  PAT10001|
|  CLM1002|   CUST1002|         ACCIDENT|      $2,500|  2026/01/07|   APPROVED |  Dr. Mehta|  PAT10002|
|  clm1003|   cust1003|   Vehicle Damage|       75000|  08-01-2026|     Pending|    Dr. Roy|  PAT10003|
| CLM1004 |   CUST1004|            THEFT|        ₹85K|  2026-01-10|    Rejected|   Dr. Khan|  PAT10004|
|  CLM1005|  CUST1005 |        Maternity|        1.5L|  2026-01-11|    approved|    Dr. Das|  PAT10005|
|  CLM1006|   cust1006|  HOSPITALIZATION|   ₹2,00,000|  2026-01-12|    Approved| Dr. Sharma|  PAT10006|
|  CLM1007|   CUST1007|        accident |       35000|  2026-01-

#*Step 2: Creating bronze data*

In [ ]:
bronze_delta_path = "/content/delta/project_insurance_claims/bronze"

In [ ]:
dirty_claims_df.write.format("delta").mode("overwrite").save(bronze_delta_path)

#*Step 4: Reading back bronze delta file*

In [ ]:
bronze_claims_df = spark.read.format("delta").load(bronze_delta_path)

In [ ]:
bronze_claims_df.show()

+--------+-----------+---------------+------------+-------------+------------+-----------+----------+
|Claim_ID|Customer_ID|     Claim_Type|Claim_Amount|   Claim_Date|Claim_Status|Doctor_Name|Patient_ID|
+--------+-----------+---------------+------------+-------------+------------+-----------+----------+
| CLM1039|   CUST1039|            OPD|     ₹12,500|   2026.02.12|    APPROVED|    Dr. Roy|  PAT10039|
| CLM1040|   CUST1040|        SURGERY|     ₹95,000|not available|    Approved|  Dr. Mehta|  PAT10040|
| CLM1041|   CUST1041|hospitalization|         ₹1L|   2026-02-14|    approved| Dr. Sharma|  PAT10041|
| CLM1042|   CUST1042|Hospitalisation|       ₹1.2L|   2026-02-15|    Approved| Dr. Sharma|  PAT10042|
| CLM1043|   CUST1043| vehicle damage|        ₹75K|   2026-02-16|     Pending|    Dr. Roy|  PAT10043|
| CLM1044|   CUST1044| VEHICLE-DAMAGE|       80000|   2026-02-17|    approved|   Dr. Khan|  PAT10044|
| CLM1045|   CUST1045|  FLIGHT DELAY |        €900|   2026-02-18|    Rejected|    

#*Step 5: Silver Layer Transformation*

#A. Creating a backup dataset

In [ ]:
backup_claims_df = bronze_claims_df

#B. Cleaning bronze data

In [ ]:
claim_id_format = r"^CLM\d{4}$"
cust_id_format = r"^CUST[_-]?\d{4}$"
claim_types = ["HOSPITALIZATION", "ACCIDENT", "VEHICLE DAMAGE", "THEFT",
               "MATERNITY", "DEATH", "BAGGAGE LOSS", "FLIGHT DELAY",
               "OPD", "MATURITY", "SURGERY"]
amount_format = r"^\d+\.\d+$"
claim_statuses = ["APPROVED", "PENDING", "REJECTED"]
pat_id_format = r"^PAT-?\d{5}$"

In [ ]:
silver_claims_df = (bronze_claims_df
                    .withColumn("raw_Claim_ID", sf.col("Claim_ID"))
                    .withColumn("Claim_ID",
                                sf.upper(sf.trim("Claim_ID")))
                    .withColumn("valid_Claim_ID",
                                sf.col("Claim_ID").rlike(claim_id_format))
                    .withColumn("Claim_ID",
                                sf.when(sf.col("valid_Claim_ID"), sf.col("Claim_ID"))
                                .otherwise(sf.lit(None)))
                    .withColumn("raw_Customer_ID", sf.col("Customer_ID"))
                    .withColumn("Customer_ID", sf.upper(sf.trim("Customer_ID")))
                    .withColumn("valid_Customer_ID",
                                sf.col("Customer_ID").rlike(cust_id_format))
                    .withColumn("Customer_ID",
                                sf.when(sf.col("valid_Customer_ID"),
                                        sf.col("Customer_ID"))
                                .otherwise(sf.lit(None)))
                    .withColumn("raw_Claim_Type", sf.col("Claim_Type"))
                    .withColumn("Claim_Type",
                                sf.regexp_replace(sf.upper(sf.trim("Claim_Type")),
                                                  r"\s+", " "))
                    .withColumn("valid_Claim_Type",
                                sf.col("Claim_Type").isin(claim_types))
                    .withColumn("Claim_Type",
                                sf.when(sf.col("valid_Claim_Type"), sf.col("Claim_Type"))
                                .otherwise(sf.lit(None)))
                    .withColumn("raw_Claim_Amount", sf.col("Claim_Amount"))
                    .withColumn("Claim_Amount_String",
                                sf.regexp_replace(
                                    sf.upper(sf.trim("Claim_Amount")),
                                    r"(?:[-$/,€₹ ]|RS\.|RS|INR|USD|ONLY)", ""))
                    .withColumn("valid_Claim_Amount",
                                sf.col("Claim_Amount_String")
                                .rlike(r"^\d+(?:\.\d+)?(?:K|L|LAKH)?$"))
                    .withColumn("Claim_Amount",
                                sf.when(sf.col("valid_Claim_Amount"),
                                        sf.when(
                                            sf.col("Claim_Amount_String").rlike(r"K$"),
                                            sf.regexp_extract(
                                                sf.col("Claim_Amount_String"),
                                                r"^(\d+(?:\.\d+)?)K$", 1)
                                            .try_cast("double") * 1000)
                                        .when(
                                            sf.col("Claim_Amount_String")
                                            .rlike(r"(?:L|LAKH)$"),
                                            sf.regexp_extract(
                                                sf.col("Claim_Amount_String"),
                                                r"^(\d+(?:\.\d+)?)(?:L|LAKH)$", 1)
                                            .try_cast("double") * 100000)
                                        .otherwise(
                                            sf.col("Claim_Amount_String")
                                            .try_cast("double")))
                                .otherwise(sf.lit(None)))
                    .withColumn("raw_Claim_Date", sf.col("Claim_Date"))
                    .withColumn("Claim_Date", sf.trim("Claim_Date"))
                    .withColumn("Claim_Date",
                                sf.coalesce(
                                    sf.try_to_timestamp("Claim_Date",
                                                        sf.lit("yyyy-MM-dd"))
                                    .try_cast("date"),
                                    sf.try_to_timestamp("Claim_Date",
                                                        sf.lit("yyyy/MM/dd"))
                                    .try_cast("date"),
                                    sf.try_to_timestamp("Claim_Date",
                                                        sf.lit("dd-MM-yyyy"))
                                    .try_cast("date"),
                                    sf.try_to_timestamp("Claim_Date",
                                                        sf.lit("dd/MM/yyyy"))
                                    .try_cast("date"),
                                    sf.try_to_timestamp("Claim_Date",
                                                        sf.lit("MM-dd-yyyy"))
                                    .try_cast("date"),
                                    sf.try_to_timestamp("Claim_Date",
                                                        sf.lit("yyyy.MM.dd"))
                                    .try_cast("date"),
                                    sf.try_to_timestamp("Claim_Date",
                                                        sf.lit("dd/MM/yy"))
                                    .try_cast("date")))
                    .withColumn("valid_Claim_Date",
                                      sf.col("Claim_Date").isNotNull())
                    .withColumn("raw_Claim_Status", sf.col("Claim_Status"))
                    .withColumn("Claim_Status", sf.upper(sf.trim("Claim_Status")))
                    .withColumn("valid_Claim_Status",
                                sf.col("Claim_Status").isin(claim_statuses))
                    .withColumn("Claim_Status",
                                sf.when(sf.col("valid_Claim_Status"),
                                        sf.col("Claim_Status"))
                                .otherwise(sf.lit(None)))
                    .withColumn("Doctor_Name",
                                sf.initcap(
                                    sf.regexp_replace(
                                        sf.regexp_replace(sf.trim("Doctor_Name"),
                                                          r"\s+", ""),
                                        r"(?i)^dr\.?\s*", "Dr ")))
                    .withColumn("raw_Patient_ID", sf.col("Patient_ID"))
                    .withColumn("Patient_ID", sf.upper(sf.trim("Patient_ID")))
                    .withColumn("valid_Patient_ID",
                                sf.col("Patient_ID").rlike(pat_id_format))
                    .withColumn("Patient_ID",
                                sf.when(sf.col("Patient_ID").rlike(pat_id_format),
                                        sf.col("Patient_ID"))
                                .otherwise(sf.lit(None)))
                    .dropDuplicates(["Claim_ID"])
                    )

In [ ]:
silver_claims_df.show(1)

+--------+-----------+----------+------------+----------+------------+-----------+----------+------------+--------------+---------------+-----------------+--------------+----------------+----------------+-------------------+------------------+--------------+----------------+----------------+------------------+--------------+----------------+
|Claim_ID|Customer_ID|Claim_Type|Claim_Amount|Claim_Date|Claim_Status|Doctor_Name|Patient_ID|raw_Claim_ID|valid_Claim_ID|raw_Customer_ID|valid_Customer_ID|raw_Claim_Type|valid_Claim_Type|raw_Claim_Amount|Claim_Amount_String|valid_Claim_Amount|raw_Claim_Date|valid_Claim_Date|raw_Claim_Status|valid_Claim_Status|raw_Patient_ID|valid_Patient_ID|
+--------+-----------+----------+------------+----------+------------+-----------+----------+------------+--------------+---------------+-----------------+--------------+----------------+----------------+-------------------+------------------+--------------+----------------+----------------+------------------+-

#B. Building Quarantine table

In [ ]:
silver_claims_df = (silver_claims_df
                    .withColumn("Validation_Failed",
                                ~(sf.coalesce(sf.col("valid_Claim_ID"), sf.lit(False)) &
                                sf.coalesce(sf.col("valid_Customer_ID"), sf.lit(False)) &
                                sf.coalesce(sf.col("valid_Claim_Type"), sf.lit(False)) &
                                sf.coalesce(sf.col("valid_Claim_Amount"), sf.lit(False)) &
                                sf.coalesce(sf.col("valid_Claim_Date"), sf.lit(False)) &
                                sf.coalesce(sf.col("valid_Claim_Status"), sf.lit(False)) &
                                sf.coalesce(sf.col("valid_Patient_ID"), sf.lit(False))))
                    )

In [ ]:
silver_claims_df.show(1)

+--------+-----------+----------+------------+----------+------------+-----------+----------+------------+--------------+---------------+-----------------+--------------+----------------+----------------+-------------------+------------------+--------------+----------------+----------------+------------------+--------------+----------------+-----------------+
|Claim_ID|Customer_ID|Claim_Type|Claim_Amount|Claim_Date|Claim_Status|Doctor_Name|Patient_ID|raw_Claim_ID|valid_Claim_ID|raw_Customer_ID|valid_Customer_ID|raw_Claim_Type|valid_Claim_Type|raw_Claim_Amount|Claim_Amount_String|valid_Claim_Amount|raw_Claim_Date|valid_Claim_Date|raw_Claim_Status|valid_Claim_Status|raw_Patient_ID|valid_Patient_ID|Validation_Failed|
+--------+-----------+----------+------------+----------+------------+-----------+----------+------------+--------------+---------------+-----------------+--------------+----------------+----------------+-------------------+------------------+--------------+----------------+-

#B.i. Quarantine Claim_ID

In [ ]:
quarantine_claim_id_df = (silver_claims_df
                          .filter(~sf.coalesce(sf.col("valid_Claim_ID"), sf.lit(False)))
                          .select(sf.col("raw_Claim_ID").alias("Claim_ID"),
                                  sf.lit("Claim_ID").alias("Column_Name"),
                                  sf.col("raw_Claim_ID").alias("Invalid_Value"),
                                  sf.lit("DQ001").alias("Rule_ID"),
                                  sf.lit("Invalid Claim_ID format").alias("Failure_Reason"))
                          )

In [ ]:
quarantine_claim_id_df.show(truncate=False)

+--------+-----------+-------------+-------+-----------------------+
|Claim_ID|Column_Name|Invalid_Value|Rule_ID|Failure_Reason         |
+--------+-----------+-------------+-------+-----------------------+
|CLM1033A|Claim_ID   |CLM1033A     |DQ001  |Invalid Claim_ID format|
+--------+-----------+-------------+-------+-----------------------+



#B.ii. Quarantine Customer_ID

In [ ]:
quarantine_customer_id_df = (silver_claims_df
                             .filter(~sf.coalesce(sf.col("valid_Customer_ID"), sf.lit(False)))
                             .select(sf.col("raw_Claim_ID").alias("Claim_ID"),
                                     sf.lit("Customer_ID").alias("Column_Name"),
                                     sf.col("raw_Customer_ID").alias("Invalid_Value"),
                                     sf.lit("DQ002").alias("Rule_ID"),
                                     sf.lit("Invalid Customer_ID format").alias("Failure_Reason"))
                             )

In [ ]:
quarantine_customer_id_df.show()

+--------+-----------+-------------+-------+--------------------+
|Claim_ID|Column_Name|Invalid_Value|Rule_ID|      Failure_Reason|
+--------+-----------+-------------+-------+--------------------+
| CLM1052|Customer_ID|         NULL|  DQ002|Invalid Customer_...|
+--------+-----------+-------------+-------+--------------------+



#B.iii. Quarantine Claim_Type

In [ ]:
quarantine_claim_type_df = (silver_claims_df
                            .filter(
                                ~sf.coalesce(sf.col("valid_Claim_Type"), sf.lit(False)))
                            .select(sf.col("raw_Claim_ID").alias("Claim_ID"),
                                    sf.lit("Claim_Type").alias("Column_Name"),
                                    sf.col("raw_Claim_Type").alias("Invalid_Value"),
                                    sf.lit("DQ003").alias("Rule_ID"),
                                    sf.lit("Invalid claim type").alias("Failure_Reason"))
                            )

In [ ]:
quarantine_claim_type_df.show()

+---------+-----------+-----------------+-------+------------------+
| Claim_ID|Column_Name|    Invalid_Value|Rule_ID|    Failure_Reason|
+---------+-----------+-----------------+-------+------------------+
| CLM1001 | Claim_Type| hospitalisation |  DQ003|Invalid claim type|
|  CLM1015| Claim_Type|  hospitalisation|  DQ003|Invalid claim type|
|  CLM1025| Claim_Type|     FLIGHT_DELAY|  DQ003|Invalid claim type|
|  CLM1027| Claim_Type|  MOTOR INSURANCE|  DQ003|Invalid claim type|
|  CLM1029| Claim_Type| HEALTH INSURANCE|  DQ003|Invalid claim type|
|  CLM1042| Claim_Type|  Hospitalisation|  DQ003|Invalid claim type|
|  CLM1044| Claim_Type|   VEHICLE-DAMAGE|  DQ003|Invalid claim type|
|  CLM1053| Claim_Type|             NULL|  DQ003|Invalid claim type|
|  CLM1072| Claim_Type|         theft!!!|  DQ003|Invalid claim type|
+---------+-----------+-----------------+-------+------------------+



In [ ]:
silver_claims_df.show(1)

+--------+-----------+----------+------------+----------+------------+-----------+----------+------------+--------------+---------------+-----------------+--------------+----------------+----------------+-------------------+------------------+--------------+----------------+----------------+------------------+--------------+----------------+-----------------+
|Claim_ID|Customer_ID|Claim_Type|Claim_Amount|Claim_Date|Claim_Status|Doctor_Name|Patient_ID|raw_Claim_ID|valid_Claim_ID|raw_Customer_ID|valid_Customer_ID|raw_Claim_Type|valid_Claim_Type|raw_Claim_Amount|Claim_Amount_String|valid_Claim_Amount|raw_Claim_Date|valid_Claim_Date|raw_Claim_Status|valid_Claim_Status|raw_Patient_ID|valid_Patient_ID|Validation_Failed|
+--------+-----------+----------+------------+----------+------------+-----------+----------+------------+--------------+---------------+-----------------+--------------+----------------+----------------+-------------------+------------------+--------------+----------------+-

#B.iv. Quarantine Claim Amount

In [ ]:
quarantine_claim_amount_df = (silver_claims_df
                              .filter(
                                  ~sf.coalesce(sf.col("valid_Claim_Amount"), sf.lit(False)))
                              .select(sf.col("raw_Claim_ID").alias("Claim_ID"),
                                      sf.lit("Claim_Amount").alias("Column_Name"),
                                      sf.col("raw_Claim_Amount").alias("Invalid_Value"),
                                      sf.lit("DQ004").alias("Rule_ID"),
                                      sf.lit("Invalid Claim Amount").alias("Failure_Reason"))
                              )

In [ ]:
quarantine_claim_amount_df.show()

+--------+------------+-------------+-------+--------------------+
|Claim_ID| Column_Name|Invalid_Value|Rule_ID|      Failure_Reason|
+--------+------------+-------------+-------+--------------------+
| CLM1008|Claim_Amount|      INVALID|  DQ004|Invalid Claim Amount|
| CLM1054|Claim_Amount|         NULL|  DQ004|Invalid Claim Amount|
| CLM1063|Claim_Amount|          N/A|  DQ004|Invalid Claim Amount|
+--------+------------+-------------+-------+--------------------+



#B.v. Quarantine Claim Date

In [ ]:
quarantine_claim_date_df = (silver_claims_df
                            .filter(~sf.coalesce(sf.col("valid_Claim_Date"), sf.lit(False)))
                            .select(sf.col("raw_Claim_ID").alias("Claim_ID"),
                                    sf.lit("Claim_Date").alias("Column_Name"),
                                    sf.col("raw_Claim_Date").alias("Invalid_Value"),
                                    sf.lit("DQ005").alias("Rule_ID"),
                                    sf.lit("Invalid Claim Date").alias("Failure_Reason"))
                            )

In [ ]:
quarantine_claim_date_df.show()

+--------+-----------+-------------+-------+------------------+
|Claim_ID|Column_Name|Invalid_Value|Rule_ID|    Failure_Reason|
+--------+-----------+-------------+-------+------------------+
| CLM1040| Claim_Date|not available|  DQ005|Invalid Claim Date|
| CLM1055| Claim_Date|         NULL|  DQ005|Invalid Claim Date|
+--------+-----------+-------------+-------+------------------+



#B.vi. Quarantine Claim Status

In [ ]:
silver_claims_df.show(1)

+--------+-----------+----------+------------+----------+------------+-----------+----------+------------+--------------+---------------+-----------------+--------------+----------------+----------------+-------------------+------------------+--------------+----------------+----------------+------------------+--------------+----------------+-----------------+
|Claim_ID|Customer_ID|Claim_Type|Claim_Amount|Claim_Date|Claim_Status|Doctor_Name|Patient_ID|raw_Claim_ID|valid_Claim_ID|raw_Customer_ID|valid_Customer_ID|raw_Claim_Type|valid_Claim_Type|raw_Claim_Amount|Claim_Amount_String|valid_Claim_Amount|raw_Claim_Date|valid_Claim_Date|raw_Claim_Status|valid_Claim_Status|raw_Patient_ID|valid_Patient_ID|Validation_Failed|
+--------+-----------+----------+------------+----------+------------+-----------+----------+------------+--------------+---------------+-----------------+--------------+----------------+----------------+-------------------+------------------+--------------+----------------+-

In [ ]:
quarantine_claim_status_df = (silver_claims_df
                              .filter(~sf.coalesce(sf.col("valid_Claim_Status"), sf.lit(False)))
                              .select(sf.col("raw_Claim_ID").alias("Claim_ID"),
                               sf.lit("Claim_Status").alias("Column_Name"),
                               sf.col("raw_Claim_Status").alias("Invalid_Value"),
                               sf.lit("DQ006").alias("Rule_ID"),
                               sf.lit("Invalid Claim Status").alias("Failure_Reason"))
                              )

In [ ]:
quarantine_claim_status_df.show()

+--------+------------+-------------+-------+--------------------+
|Claim_ID| Column_Name|Invalid_Value|Rule_ID|      Failure_Reason|
+--------+------------+-------------+-------+--------------------+
| CLM1046|Claim_Status|      APPROVE|  DQ006|Invalid Claim Status|
| CLM1047|Claim_Status|          REJ|  DQ006|Invalid Claim Status|
| CLM1048|Claim_Status|    In Review|  DQ006|Invalid Claim Status|
| CLM1050|Claim_Status|      unknown|  DQ006|Invalid Claim Status|
+--------+------------+-------------+-------+--------------------+



#B.vii. Quarantine Patient ID

In [ ]:
quarantine_patient_id_df = (silver_claims_df
                            .filter(~sf.coalesce(sf.col("valid_Patient_ID"), sf.lit(False)))
                            .select(sf.col("raw_Claim_ID").alias("Claim_ID"),
                               sf.lit("Patient_ID").alias("Column_Name"),
                               sf.col("raw_Patient_ID").alias("Invalid_Value"),
                               sf.lit("DQ007").alias("Rule_ID"),
                               sf.lit("Invalid Patient ID").alias("Failure_Reason"))
                            )

In [ ]:
quarantine_patient_id_df.show()

+--------+-----------+-------------+-------+--------------+
|Claim_ID|Column_Name|Invalid_Value|Rule_ID|Failure_Reason|
+--------+-----------+-------------+-------+--------------+
+--------+-----------+-------------+-------+--------------+



#C. Combining all quarantined columns

In [ ]:
quarantine_claims_df = (
    quarantine_claim_id_df
    .unionByName(quarantine_customer_id_df)
    .unionByName(quarantine_claim_type_df)
    .unionByName(quarantine_claim_amount_df)
    .unionByName(quarantine_claim_date_df)
    .unionByName(quarantine_claim_status_df)
    .unionByName(quarantine_patient_id_df)
)

In [ ]:
quarantine_claims_df.show()

+---------+------------+-----------------+-------+--------------------+
| Claim_ID| Column_Name|    Invalid_Value|Rule_ID|      Failure_Reason|
+---------+------------+-----------------+-------+--------------------+
| CLM1033A|    Claim_ID|         CLM1033A|  DQ001|Invalid Claim_ID ...|
|  CLM1052| Customer_ID|             NULL|  DQ002|Invalid Customer_...|
| CLM1001 |  Claim_Type| hospitalisation |  DQ003|  Invalid claim type|
|  CLM1015|  Claim_Type|  hospitalisation|  DQ003|  Invalid claim type|
|  CLM1025|  Claim_Type|     FLIGHT_DELAY|  DQ003|  Invalid claim type|
|  CLM1027|  Claim_Type|  MOTOR INSURANCE|  DQ003|  Invalid claim type|
|  CLM1029|  Claim_Type| HEALTH INSURANCE|  DQ003|  Invalid claim type|
|  CLM1042|  Claim_Type|  Hospitalisation|  DQ003|  Invalid claim type|
|  CLM1044|  Claim_Type|   VEHICLE-DAMAGE|  DQ003|  Invalid claim type|
|  CLM1053|  Claim_Type|             NULL|  DQ003|  Invalid claim type|
|  CLM1072|  Claim_Type|         theft!!!|  DQ003|  Invalid clai

In [ ]:
quarantine_claims_df.orderBy("Claim_ID", "Rule_ID").show(truncate=False)

+---------+------------+-----------------+-------+--------------------------+
|Claim_ID |Column_Name |Invalid_Value    |Rule_ID|Failure_Reason            |
+---------+------------+-----------------+-------+--------------------------+
| CLM1001 |Claim_Type  | hospitalisation |DQ003  |Invalid claim type        |
| CLM1015 |Claim_Type  |hospitalisation  |DQ003  |Invalid claim type        |
|CLM1008  |Claim_Amount|INVALID          |DQ004  |Invalid Claim Amount      |
|CLM1025  |Claim_Type  |FLIGHT_DELAY     |DQ003  |Invalid claim type        |
|CLM1027  |Claim_Type  |MOTOR INSURANCE  |DQ003  |Invalid claim type        |
|CLM1029  |Claim_Type  |HEALTH INSURANCE |DQ003  |Invalid claim type        |
|CLM1033A |Claim_ID    |CLM1033A         |DQ001  |Invalid Claim_ID format   |
|CLM1040  |Claim_Date  |not available    |DQ005  |Invalid Claim Date        |
|CLM1042  |Claim_Type  |Hospitalisation  |DQ003  |Invalid claim type        |
|CLM1044  |Claim_Type  |VEHICLE-DAMAGE   |DQ003  |Invalid claim 

#Step 6: Creating cleaned silver table

In [ ]:
clean_silver_claims_df = (silver_claims_df
                          .filter(~sf.col("Validation_Failed"))
                          )

In [ ]:
clean_silver_claims_df.show()

+--------+-----------+---------------+------------+----------+------------+-----------+----------+------------+--------------+---------------+-----------------+---------------+----------------+----------------+-------------------+------------------+--------------+----------------+----------------+------------------+--------------+----------------+-----------------+
|Claim_ID|Customer_ID|     Claim_Type|Claim_Amount|Claim_Date|Claim_Status|Doctor_Name|Patient_ID|raw_Claim_ID|valid_Claim_ID|raw_Customer_ID|valid_Customer_ID| raw_Claim_Type|valid_Claim_Type|raw_Claim_Amount|Claim_Amount_String|valid_Claim_Amount|raw_Claim_Date|valid_Claim_Date|raw_Claim_Status|valid_Claim_Status|raw_Patient_ID|valid_Patient_ID|Validation_Failed|
+--------+-----------+---------------+------------+----------+------------+-----------+----------+------------+--------------+---------------+-----------------+---------------+----------------+----------------+-------------------+------------------+--------------+

#Step 7: Final cleaned silver table

In [ ]:
final_silver_claims_df = (clean_silver_claims_df
                          .select("Claim_ID", "Customer_ID", "Claim_Type",
                                  "Claim_Amount", "Claim_Date", "Claim_Status",
                                  "Doctor_Name", "Patient_ID"))

In [ ]:
final_silver_claims_df.show()

+--------+-----------+---------------+------------+----------+------------+-----------+----------+
|Claim_ID|Customer_ID|     Claim_Type|Claim_Amount|Claim_Date|Claim_Status|Doctor_Name|Patient_ID|
+--------+-----------+---------------+------------+----------+------------+-----------+----------+
| CLM1002|   CUST1002|       ACCIDENT|      2500.0|2026-01-07|    APPROVED|   Dr Mehta|  PAT10002|
| CLM1003|   CUST1003| VEHICLE DAMAGE|     75000.0|2026-01-08|     PENDING|     Dr Roy|  PAT10003|
| CLM1004|   CUST1004|          THEFT|     85000.0|2026-01-10|    REJECTED|    Dr Khan|  PAT10004|
| CLM1005|   CUST1005|      MATERNITY|    150000.0|2026-01-11|    APPROVED|     Dr Das|  PAT10005|
| CLM1006|   CUST1006|HOSPITALIZATION|    200000.0|2026-01-12|    APPROVED|  Dr Sharma|  PAT10006|
| CLM1007|   CUST1007|       ACCIDENT|     35000.0|2026-01-13|     PENDING|   Dr Mehta|  PAT10007|
| CLM1009|   CUST1009|          DEATH|    500000.0|2026-01-15|    APPROVED|    Dr Khan|  PAT10009|
| CLM1010|

In [ ]:
final_silver_claims_df.printSchema()

root
 |-- Claim_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Claim_Type: string (nullable = true)
 |-- Claim_Amount: double (nullable = true)
 |-- Claim_Date: date (nullable = true)
 |-- Claim_Status: string (nullable = true)
 |-- Doctor_Name: string (nullable = true)
 |-- Patient_ID: string (nullable = true)



#Step 8: Storing silver data into delta

#A. Saving into delta

In [ ]:
silver_delta_path = "/content/delta/project_insurance_claims/silver"

In [ ]:
final_silver_claims_df.write.format("delta").mode("overwrite").save(silver_delta_path)

#B. Reading Silver Data from Delta

In [ ]:
silver_delta_claims_df = spark.read.format("delta").load(silver_delta_path)

In [ ]:
silver_delta_claims_df.show()

+--------+-----------+---------------+------------+----------+------------+-----------+----------+
|Claim_ID|Customer_ID|     Claim_Type|Claim_Amount|Claim_Date|Claim_Status|Doctor_Name|Patient_ID|
+--------+-----------+---------------+------------+----------+------------+-----------+----------+
| CLM1002|   CUST1002|       ACCIDENT|      2500.0|2026-01-07|    APPROVED|   Dr Mehta|  PAT10002|
| CLM1003|   CUST1003| VEHICLE DAMAGE|     75000.0|2026-01-08|     PENDING|     Dr Roy|  PAT10003|
| CLM1004|   CUST1004|          THEFT|     85000.0|2026-01-10|    REJECTED|    Dr Khan|  PAT10004|
| CLM1005|   CUST1005|      MATERNITY|    150000.0|2026-01-11|    APPROVED|     Dr Das|  PAT10005|
| CLM1006|   CUST1006|HOSPITALIZATION|    200000.0|2026-01-12|    APPROVED|  Dr Sharma|  PAT10006|
| CLM1007|   CUST1007|       ACCIDENT|     35000.0|2026-01-13|     PENDING|   Dr Mehta|  PAT10007|
| CLM1009|   CUST1009|          DEATH|    500000.0|2026-01-15|    APPROVED|    Dr Khan|  PAT10009|
| CLM1010|

#Step 9: Gold Layer Transformation

#A. Gold #1 — Claims Summary by Claim Type.

In [ ]:
gold_claim_type_summary_df = (final_silver_claims_df
                              .groupBy("Claim_Type")
                              .agg(
                                  sf.count("*").alias("Total_Claims"),
                                  sf.round(sf.sum("Claim_Amount"), 2)
                                  .alias("Total_Claim_Amount"),
                                  sf.round(sf.avg("Claim_Amount"), 2)
                                  .alias("Avg_Claim_Amount"),
                                  sf.sum(
                                      sf.when(sf.col("Claim_Status") ==
                                              "APPROVED", 1).otherwise(0))
                                  .alias("Approved_Claims"),
                                  sf.sum(
                                      sf.when(sf.col("Claim_Status") == "PENDING", 1)
                                      .otherwise(0)).alias("Pending_Claims"),
                                  sf.sum(
                                      sf.when(sf.col("Claim_Status") == "REJECTED", 1)
                                      .otherwise(0)).alias("Rejected_Claims"))
                              .orderBy("Claim_Type")
                              )

In [ ]:
gold_claim_type_summary_df.show()

+---------------+------------+------------------+----------------+---------------+--------------+---------------+
|     Claim_Type|Total_Claims|Total_Claim_Amount|Avg_Claim_Amount|Approved_Claims|Pending_Claims|Rejected_Claims|
+---------------+------------+------------------+----------------+---------------+--------------+---------------+
|       ACCIDENT|          12|          690500.5|        57541.71|              8|             4|              0|
|   BAGGAGE LOSS|           2|           80000.0|         40000.0|              0|             0|              2|
|          DEATH|           4|         2200000.0|        550000.0|              3|             0|              1|
|   FLIGHT DELAY|           2|            2100.0|          1050.0|              1|             0|              1|
|HOSPITALIZATION|           4|          650000.0|        162500.0|              4|             0|              0|
|      MATERNITY|           4|          625000.0|        156250.0|              4|      

#B. Gold #2: Claims Summary by Status

In [ ]:
gold_claim_status_summary_df = (final_silver_claims_df
                                .groupBy("Claim_Status")
                                .agg(sf.count("*").alias("Total_Claims"),
                                     sf.sum("Claim_Amount").alias("Total_Claim_Amount"),
                                     sf.round(sf.avg("Claim_Amount"), 2)
                                     .alias("Avg_Claim_Amount"))
                                .orderBy("Claim_Status")
                                )

In [ ]:
gold_claim_status_summary_df.show()

+------------+------------+------------------+----------------+
|Claim_Status|Total_Claims|Total_Claim_Amount|Avg_Claim_Amount|
+------------+------------+------------------+----------------+
|    APPROVED|          34|         4596700.5|       135197.07|
|     PENDING|          13|         968500.75|        74500.06|
|    REJECTED|           6|          650900.0|       108483.33|
+------------+------------+------------------+----------------+



#C. Gold #3 — Claims by Month

In [ ]:
final_silver_claims_df.show(1)

+--------+-----------+----------+------------+----------+------------+-----------+----------+
|Claim_ID|Customer_ID|Claim_Type|Claim_Amount|Claim_Date|Claim_Status|Doctor_Name|Patient_ID|
+--------+-----------+----------+------------+----------+------------+-----------+----------+
| CLM1002|   CUST1002|  ACCIDENT|      2500.0|2026-01-07|    APPROVED|   Dr Mehta|  PAT10002|
+--------+-----------+----------+------------+----------+------------+-----------+----------+
only showing top 1 row


In [ ]:
gold_claim_monthly_summary_df = (final_silver_claims_df
                                 .groupBy(sf.date_format("Claim_Date", "yyyy-MM")
                                 .alias("Month"))
                                 .agg(sf.count("*").alias("Claim_Count"),
                                      sf.sum("Claim_Amount")
                                      .alias("Total_Claim_Amount"),
                                      sf.round(sf.avg("Claim_Amount"), 2)
                                      .alias("Avg_Claim_Amount"))
                                 .orderBy("Month")
                                 )

In [ ]:
gold_claim_monthly_summary_df.show()

+-------+-----------+------------------+----------------+
|  Month|Claim_Count|Total_Claim_Amount|Avg_Claim_Amount|
+-------+-----------+------------------+----------------+
|2026-01|         22|        2672200.75|       121463.67|
|2026-02|         14|         1453400.0|       103814.29|
|2026-03|         17|         2090500.5|       122970.62|
+-------+-----------+------------------+----------------+



#D. Gold #4: Claims by Claim Type and Status

In [ ]:
gold_claim_by_type_status_df = (final_silver_claims_df
                                .groupBy("Claim_Type", "Claim_Status")
                                .agg(sf.count("*").alias("Claim_Count"),
                                     sf.sum("Claim_Amount")
                                     .alias("Total_Claim_Amount"),
                                     sf.round(sf.avg("Claim_Amount"), 2)
                                      .alias("Avg_Claim_Amount"))
                                .orderBy("Claim_Type", "Claim_Status")
                                )

In [ ]:
gold_claim_by_type_status_df.show()

+---------------+------------+-----------+------------------+----------------+
|     Claim_Type|Claim_Status|Claim_Count|Total_Claim_Amount|Avg_Claim_Amount|
+---------------+------------+-----------+------------------+----------------+
|       ACCIDENT|    APPROVED|          8|          483000.5|        60375.06|
|       ACCIDENT|     PENDING|          4|          207500.0|         51875.0|
|   BAGGAGE LOSS|    REJECTED|          2|           80000.0|         40000.0|
|          DEATH|    APPROVED|          3|         1800000.0|        600000.0|
|          DEATH|    REJECTED|          1|          400000.0|        400000.0|
|   FLIGHT DELAY|    APPROVED|          1|            1200.0|          1200.0|
|   FLIGHT DELAY|    REJECTED|          1|             900.0|           900.0|
|HOSPITALIZATION|    APPROVED|          4|          650000.0|        162500.0|
|      MATERNITY|    APPROVED|          4|          625000.0|        156250.0|
|       MATURITY|    APPROVED|          1|          

#E. Gold #5: Claims Performance by Doctor

In [ ]:
final_silver_claims_df.show(1)

+--------+-----------+----------+------------+----------+------------+-----------+----------+
|Claim_ID|Customer_ID|Claim_Type|Claim_Amount|Claim_Date|Claim_Status|Doctor_Name|Patient_ID|
+--------+-----------+----------+------------+----------+------------+-----------+----------+
| CLM1002|   CUST1002|  ACCIDENT|      2500.0|2026-01-07|    APPROVED|   Dr Mehta|  PAT10002|
+--------+-----------+----------+------------+----------+------------+-----------+----------+
only showing top 1 row


In [ ]:
gold_claim_doctors_df = (final_silver_claims_df
                          .groupBy("Doctor_Name")
                          .agg(sf.count("*").alias("Claim_Count"),
                               sf.sum("Claim_Amount")
                               .alias("Total_Claim_Amount"),
                               sf.sum(sf.when(sf.col("Claim_Status") == "APPROVED",
                                       sf.col("Claim_Amount")).otherwise(sf.lit(0)))
                               .alias("Approved_Claims"),
                               sf.sum(sf.when(sf.col("Claim_Status") == "REJECTED",
                                       sf.col("Claim_Amount")).otherwise(sf.lit(0)))
                               .alias("Rejected_Claims"),
                               sf.sum(sf.when(sf.col("Claim_Status") == "PENDING",
                                       sf.col("Claim_Amount")).otherwise(sf.lit(0)))
                               .alias("Pending_Claims"))
                          .orderBy("Doctor_Name")
                          )

In [ ]:
gold_claim_doctors_df.show()

+-----------+-----------+------------------+---------------+---------------+--------------+
|Doctor_Name|Claim_Count|Total_Claim_Amount|Approved_Claims|Rejected_Claims|Pending_Claims|
+-----------+-----------+------------------+---------------+---------------+--------------+
|       NULL|          1|          150000.0|       150000.0|            0.0|           0.0|
|     Dr Das|         10|         1013400.0|       730000.0|          900.0|      282500.0|
|    Dr Khan|         11|         1012500.0|       695000.0|       250000.0|       67500.0|
|   Dr Mehta|         10|         1362500.0|      1222500.0|            0.0|      140000.0|
|     Dr Roy|         10|        1462701.25|       784200.5|       400000.0|     278500.75|
|  Dr Sharma|         11|         1215000.0|      1015000.0|            0.0|      200000.0|
+-----------+-----------+------------------+---------------+---------------+--------------+



#F. Gold #6 — High-Value Claims Analysis

In [ ]:
gold_high_value_claim_df = (final_silver_claims_df
                            .filter(sf.col("Claim_Amount") > 100000)
                            .groupBy("Claim_Type")
                            .agg(sf.count("*").alias("High_Value_Claim_Count"),
                                 sf.sum("Claim_Amount")
                                 .alias("Total_High_Value_Amount"),
                                 sf.round(sf.avg("Claim_Amount"), 2)
                                 .alias("Avg_High_Value_Amount"))
                            .orderBy("Claim_Type")
                            )

In [ ]:
gold_high_value_claim_df.show()

+---------------+----------------------+-----------------------+---------------------+
|     Claim_Type|High_Value_Claim_Count|Total_High_Value_Amount|Avg_High_Value_Amount|
+---------------+----------------------+-----------------------+---------------------+
|       ACCIDENT|                     3|               370500.5|            123500.17|
|          DEATH|                     4|              2200000.0|             550000.0|
|HOSPITALIZATION|                     3|               550000.0|            183333.33|
|      MATERNITY|                     4|               625000.0|             156250.0|
|       MATURITY|                     3|               630000.0|             210000.0|
| VEHICLE DAMAGE|                     1|               120000.0|             120000.0|
+---------------+----------------------+-----------------------+---------------------+



#G. Gold #7: Claims by Doctor and Claim Status

In [ ]:
gold_claims_doctor_status_df = (final_silver_claims_df
                               .groupBy("Doctor_Name", "Claim_Status")
                               .agg(sf.count("*").alias("Claim_Count"),
                                    sf.sum("Claim_Amount")
                                    .alias("Total_Claim_Amount"),
                                    sf.round(sf.avg("Claim_Amount"), 2)
                                    .alias("Avg_Claim_Amount"))
                               .orderBy("Doctor_Name", "Claim_Status")
                               )

In [ ]:
gold_claims_doctor_status_df.show()

+-----------+------------+-----------+------------------+----------------+
|Doctor_Name|Claim_Status|Claim_Count|Total_Claim_Amount|Avg_Claim_Amount|
+-----------+------------+-----------+------------------+----------------+
|       NULL|    APPROVED|          1|          150000.0|        150000.0|
|     Dr Das|    APPROVED|          6|          730000.0|       121666.67|
|     Dr Das|     PENDING|          3|          282500.0|        94166.67|
|     Dr Das|    REJECTED|          1|             900.0|           900.0|
|    Dr Khan|    APPROVED|          5|          695000.0|        139000.0|
|    Dr Khan|     PENDING|          2|           67500.0|         33750.0|
|    Dr Khan|    REJECTED|          4|          250000.0|         62500.0|
|   Dr Mehta|    APPROVED|          7|         1222500.0|       174642.86|
|   Dr Mehta|     PENDING|          3|          140000.0|        46666.67|
|     Dr Roy|    APPROVED|          5|          784200.5|        156840.1|
|     Dr Roy|     PENDING

#H. Gold #8 — Claims Approval Performance

In [ ]:
gold_claim_types_df = (final_silver_claims_df
                       .groupBy("Claim_Type")
                       .agg(sf.count("*").alias("Total_Claims"),
                            sf.count(sf.when(sf.col("Claim_Status") == "APPROVED",
                                           sf.col("Claim_ID")))
                            .alias("Approved_Claims"),
                            sf.count(sf.when(sf.col("Claim_Status") == "REJECTED",
                                           sf.col("Claim_ID")))
                            .alias("Rejected_Claims"),
                            sf.count(sf.when(sf.col("Claim_Status") == "PENDING",
                                           sf.col("Claim_ID")))
                            .alias("Pending_Claims"))
                       .withColumn("Approval_Rate",
                                   sf.round((sf.col("Approved_Claims") /
                                    sf.col("Total_Claims"))*100, 2))
                       .orderBy("Claim_Type")
                       )

In [ ]:
gold_claim_types_df.show()

+---------------+------------+---------------+---------------+--------------+-------------+
|     Claim_Type|Total_Claims|Approved_Claims|Rejected_Claims|Pending_Claims|Approval_Rate|
+---------------+------------+---------------+---------------+--------------+-------------+
|       ACCIDENT|          12|              8|              0|             4|        66.67|
|   BAGGAGE LOSS|           2|              0|              2|             0|          0.0|
|          DEATH|           4|              3|              1|             0|         75.0|
|   FLIGHT DELAY|           2|              1|              1|             0|         50.0|
|HOSPITALIZATION|           4|              4|              0|             0|        100.0|
|      MATERNITY|           4|              4|              0|             0|        100.0|
|       MATURITY|           3|              1|              0|             2|        33.33|
|            OPD|           5|              4|              0|             1|   

#I. Gold #9 — Monthly Claim Status Analysis

In [ ]:
gold_claim_monthly_status_summary_df = (final_silver_claims_df
                                        .groupBy(sf.date_format("Claim_Date", "yyyy-MM")
                                        .alias("Month"), "Claim_Status")
                                        .agg(sf.count("*").alias("Claim_Count"),
                                             sf.sum("Claim_Amount")
                                             .alias("Total_Claim_Amount"),
                                             sf.round(sf.avg("Claim_Amount"), 2)
                                             .alias("Avg_Claim_Amount"))
                                        .orderBy("Claim_Status", "Month")
                                        )

In [ ]:
gold_claim_monthly_status_summary_df.show()

+-------+------------+-----------+------------------+----------------+
|  Month|Claim_Status|Claim_Count|Total_Claim_Amount|Avg_Claim_Amount|
+-------+------------+-----------+------------------+----------------+
|2026-01|    APPROVED|         13|         2063700.0|       158746.15|
|2026-02|    APPROVED|          9|          707500.0|        78611.11|
|2026-03|    APPROVED|         12|         1825500.5|       152125.04|
|2026-01|     PENDING|          5|         358500.75|        71700.15|
|2026-02|     PENDING|          3|          345000.0|        115000.0|
|2026-03|     PENDING|          5|          265000.0|         53000.0|
|2026-01|    REJECTED|          4|          250000.0|         62500.0|
|2026-02|    REJECTED|          2|          400900.0|        200450.0|
+-------+------------+-----------+------------------+----------------+



#J. Gold #10 — Top 5 Highest-Value Claims

In [ ]:
gold_claim_top_5_df = (final_silver_claims_df
                      .orderBy(sf.col("Claim_Amount").desc())
                      .limit(5)
                      .select("Claim_ID", "Customer_ID", "Claim_Type", "Claim_Amount",
                              "Claim_Date", "Claim_Status", "Doctor_Name")
                      )

In [ ]:
gold_claim_top_5_df.show()

+--------+-----------+----------+------------+----------+------------+-----------+
|Claim_ID|Customer_ID|Claim_Type|Claim_Amount|Claim_Date|Claim_Status|Doctor_Name|
+--------+-----------+----------+------------+----------+------------+-----------+
| CLM1061|   CUST1061|     DEATH|    750000.0|2026-03-06|    APPROVED|   Dr Mehta|
| CLM1022|   CUST1022|     DEATH|    550000.0|2026-01-27|    APPROVED|     Dr Roy|
| CLM1009|   CUST1009|     DEATH|    500000.0|2026-01-15|    APPROVED|    Dr Khan|
| CLM1035|   CUST1035|     DEATH|    400000.0|2026-02-09|    REJECTED|     Dr Roy|
| CLM1075|   CUST1075|  MATURITY|    250000.0|2026-03-20|    APPROVED|  Dr Sharma|
+--------+-----------+----------+------------+----------+------------+-----------+

